# Setup

In [ ]:
# base
import os
import sys
import gc
import re
import warnings
import logging
import pickle
from time import ctime, time
from datetime import timedelta
from collections import Counter
import itertools

# data manipulation
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from rpy2.robjects.conversion import localconverter

# single cell
import anndata as ad
import scanpy as sc

# custom
sys.path.insert(0, '../..')
from single_cell.R import *
from single_cell.preprocess import *
from single_cell.plot import *
from single_cell.analysis import *
from utils import *

warnings.simplefilter("ignore", FutureWarning)
warnings.simplefilter("ignore", UserWarning)
warnings.simplefilter("ignore", RuntimeWarning)
warnings.simplefilter("ignore", pd.errors.DtypeWarning)
warnings.simplefilter("ignore", pd.errors.PerformanceWarning)
mlogger = logging.getLogger("matplotlib")
mlogger.setLevel(logging.WARNING)

CORES = 10
DATADIR = Path("../../../data") / "processed" / "single_cell"
REFDIR = Path("../../../references")
MAIN_DIR = DATADIR / "combined"
SUBSETS_DIR = MAIN_DIR / "subsets"

METADATA = [
    "Diet",
    "Age",
    "Depot",
    "Sex",
]
DOUBLETMETHODS = ["scDblFinder", "DoubletFinder", "doubletdetection", "scrublet"]
GLOBAL_INT_KEY = "global_INT_scvi-hvg-Identifier"

converter = get_converter()
%load_ext rpy2.ipython
%matplotlib inline
# R_preload()
mpl.rcdefaults()
gc.collect()

In [ ]:
CELLTYPE_ADDON = "macro"
CLUSTER_KEY = "leiden_macro"
DE_KEY = "macro_DEGs"
SELECT_RES = [0.2, 0.4, 0.6]
INT_KEY = f"{CELLTYPE_ADDON}_INT_harmony-hvg-Identifier"
contam_resolutions = np.arange(1, 6) / 5
resolutions = np.arange(1, 10) / 10

# Load Data

In [ ]:
adata = sc.read_h5ad(MAIN_DIR / "eWAT_Male.h5ad")
adata

# Macrophages

In [ ]:
# subset
adata_macro = adata[adata.obs["celltype"].isin(["Macrophage", "Dendritic"])].copy()
print(adata_macro)
del adata_macro.uns, adata_macro.varm, adata_macro.obsp
Filter_QC(adata_macro)

### Clean contaminants

In [ ]:
# cluster
cluster = False
resolutions = np.arange(1, 10) / 10

if cluster is True:
    Cluster(adata_macro, CLUSTER_KEY, resolutions, neighbor_key="neighbors_macro")

# plot clusters
for embedding in [f"UMAP_{CELLTYPE_ADDON}", f"LocalMAP_{CELLTYPE_ADDON}"]:
    r, c = 3, 3
    f, axs = plt.subplots(r, c, figsize=(8 * c, 6 * r), layout="constrained")
    axs = axs.flatten()

    for i, res in enumerate(tqdm(resolutions)):
        res_key = f"{CLUSTER_KEY}_{res}"
        cluster_c = color_gen(adata_macro.obs[res_key].cat.categories)
        adata_macro.obs[res_key] = (
            adata_macro.obs[res_key].astype(int).astype("category")
        )

        sc.pl.embedding(
            adata_macro,
            basis=embedding,
            color=res_key,
            ax=axs[i],
            show=False,
            legend_loc="on data",
            legend_fontoutline=2,
            legend_fontsize=15,
            palette=cluster_c,
        )
        axs[i].annotate(
            f"n = {adata_macro.shape[0]}",
            size=15,
            fontweight="bold",
            xy=(0.98, 0.02),
            xycoords="axes fraction",
            horizontalalignment="right",
            verticalalignment="bottom",
        )

In [ ]:
# Dendritic Cells
for marker in ["Flt3"]:
    f = clustree(
        adata_macro,
        [f"{CLUSTER_KEY}_{res}" for res in resolutions],
        title=f"likely Dendritic Cell ({marker})",
        edge_weight_threshold=0.05,
        node_color_gene=marker,
        node_color_gene_use_raw=False,
        node_colormap="Reds",
        show_colorbar=True,
        x_spacing=1.5,
    )

    # f.set_size_inches(7, 9)

# Mono/Macs
for marker in ["Adgre1", "Mafb"]:
    f = clustree(
        adata_macro,
        [f"{CLUSTER_KEY}_{res}" for res in resolutions],
        title=f"likely NOT Macrophage ({marker})",
        edge_weight_threshold=0.05,
        node_color_gene=marker,
        node_color_gene_use_raw=False,
        node_colormap="Reds_r",
        show_colorbar=True,
        x_spacing=1.5,
    )

# Mono/Macs
for marker in ["Igkc"]:
    f = clustree(
        adata_macro,
        [f"{CLUSTER_KEY}_{res}" for res in resolutions],
        title=f"B Cell ({marker})",
        edge_weight_threshold=0.05,
        node_color_gene=marker,
        node_color_gene_use_raw=False,
        node_colormap="Reds",
        show_colorbar=True,
        x_spacing=1.5,
    )

In [ ]:
markers = {
    "Adipocyte": ["Adipoq", "Plin4", "Retn", "Ucp1"],
    "Fibroblast": ["Pdgfra", "Dcn"],
    # "Uncommitted FBs": ["Dpp4", "Cd55", "Pi16", "Aldh1a3"],
    # "Committed FBs": ["Pparg", "Icam1", "Cd36"],
    # "Areg (iWAT)" : ["F3", "Clec11a"],
    # "Endothelial": ["Pecam1", "Cdh5"],
    # "Lymphatic Endo": ["Prox1", "Lyve1"],
    # "Arterial Endo": ["Hey1", "Gkn3"],
    # "Venous Endo": ["Vcam1", "Ackr1"],
    # "Capillary Endo": ["Rgcc", "Car4"],
    # "Mesothelial": ["Msln", "Krt19", "Wt1", "Upk3b"],
    # "Pericyte" : ["Steap4", "Enpep"],
    # "SMC": ["Myocd", "Myh11", "Acta2"],
    "Mono/Macs": ["Adgre1", "Mafb"],
    # "MonoMac subsets": ["Lyve1", "Selenop", "C1qa", "Cd9", "Trem2", "Lpl", "Ccl3", "Cxcl3", "Plac8"],
    "Dendritic cell": ["Flt3"],
    # "Mast cell": ["Il1rl1", "Cpa3", "Kit"],
    # "Neutrophil": ["Csf3r", "S100a9"],
    # "NK cell": ["Klrd1", "Xcl1", "Klrb1c"],
    "T cell": ["Cd3d", "Cd3e", "Skap1"],
    # "T cell subsets" : ["Cd4", "Cd8b1", "Cxcr3", "Ccr7", "Sell", "Klrg1", "Il7", "Foxp3"],
    "B cell": ["Ms4a1", "Igkc", "Cd79a", "Cd79b"],
    # "Plasmablast" : ["Jchain"],
    # "Others": ["Dcdc2a", "Erbb4"],
}

markers_list = pd.Series([b for a in markers for b in markers[a]])
markers_list[~markers_list.isin(adata_macro.var_names)]

for res in [0.8]:
    res_key = f"{CLUSTER_KEY}_{res:.1f}"
    clear_uns(adata_macro, "colors")
    plot_violinplot(adata_macro, res_key, markers)

In [ ]:
# remove likely doublets
adata_macro = adata_macro[
    ~adata_macro.obs[f"{CLUSTER_KEY}_0.8"].isin([6, 15, 16])
].copy()
print(adata_macro.shape)

In [ ]:
# separate Dendritic Cells
adata_dcs = adata_macro[adata_macro.obs[f"{CLUSTER_KEY}_0.8"].isin([11, 13])].copy()
print(adata_dcs.shape)
clear_adata(adata_dcs, [CELLTYPE_ADDON, "colors", "dendrogram"])
adata_dcs.write_zarr(SUBSETS_DIR / "dendritic")

In [ ]:
adata_macro = adata_macro[~adata_macro.obs[f"{CLUSTER_KEY}_0.8"].isin([11, 13])].copy()

### Try reintegration

In [ ]:
# compare all integrations
r, c = 2, 2
f, axs = plt.subplots(r, c, figsize=(8 * c, 6 * r), layout="constrained")
axs = axs.flatten()
int_check = "Dataset"

for i, dr_key in enumerate(
    [
        f"UMAP_{CELLTYPE_ADDON}_INT_{method}-{modifier}-{batch}"
        for method in ["harmony"]
        for batch in ["Dataset", "Identifier"]
        for modifier in ["hvg", "all"]
    ]
):
    sc.pl.embedding(
        adata_macro,
        basis=dr_key,
        color=int_check,
        alpha=1,
        ax=axs[i],
        show=False,
        legend_loc="none" if i < len(axs) - 1 else "right margin",
        palette=create_palette(8)
    )

    if i == len(axs) - 1:
        legend_info = axs[-1].get_legend_handles_labels()
        f, ax = plt.subplots()
        ax.legend(*legend_info)
        ax.axis("off")

        axs[-1].get_legend().remove()

In [ ]:
method, modifier, batch = "harmony", "hvg", "Dataset"
int_key = f"{CELLTYPE_ADDON}_INT_{method}-{modifier}-{batch}"

check_integration(
    adata_macro, "Dataset",
    embeddings=[f"UMAP_{int_key}", f"LocalMAP_{int_key}"],
    f=plt.figure(figsize=(18,10), layout="constrained"),
    nrow=3)

method, modifier, batch = "harmony", "hvg", "Identifier"
int_key = f"{CELLTYPE_ADDON}_INT_{method}-{modifier}-{batch}"

check_integration(
    adata_macro, "Dataset",
    embeddings=[f"UMAP_{int_key}", f"LocalMAP_{int_key}"],
    f=plt.figure(figsize=(18,10), layout="constrained"),
    nrow=3)

### Clean contaminants round 2

In [ ]:
# cluster
cluster = False

if cluster is True:
    Cluster(adata_macro, CLUSTER_KEY, contam_resolutions, neighbor_key=f"neighbors_{INT_KEY}")

# plot clusters
for embedding in [f"UMAP_{INT_KEY}", f"LocalMAP_{INT_KEY}"]:
    r, c = 2, 3
    f, axs = plt.subplots(r, c, figsize=(8 * c, 6 * r), layout="constrained")
    axs = axs.flatten()

    for i, res in enumerate(tqdm(contam_resolutions)):
        res_key = f"{CLUSTER_KEY}_{res}"
        cluster_c = color_gen(adata_macro.obs[res_key].cat.categories)
        adata_macro.obs[res_key] = (
            adata_macro.obs[res_key].astype(int).astype("category")
        )

        sc.pl.embedding(
            adata_macro,
            basis=embedding,
            color=res_key,
            ax=axs[i],
            show=False,
            legend_loc="on data",
            legend_fontoutline=2,
            legend_fontsize=15,
            palette=cluster_c,
        )
        axs[i].annotate(
            f"n = {adata_macro.shape[0]}",
            size=15,
            fontweight="bold",
            xy=(0.98, 0.02),
            xycoords="axes fraction",
            horizontalalignment="right",
            verticalalignment="bottom",
        )

f = plot_cluster_trees(
    adata_macro,
    [f"{CLUSTER_KEY}_{res}" for res in contam_resolutions],
    threshold=0.03,
    node_size_range=(100, 2000),
    x_spacing=2,
    y_spacing=2,
    show_fraction=True,
)

In [ ]:
celltype_markers = {
    "Adipocyte": ["Adipoq"],
    "Fibroblast": ["Pdgfra"],
    "Mesothlial": ["Wt1"],
    "Endothelial": ["Cdh5"],
    "Smooth Muscle Cell": ["Myocd"],
    "Pericyte": ["Enpep"],
    "Epithelial 1": ["Dcdc2a"],
    "Epithelial 2": ["Erbb4"],
    "Macrophage": ["Adgre1"],
    "Dendritic Cell": ["Flt3"],
    "Mast Cell": ["Cpa3"],
    "Neutrophil": ["Csf3r"],
    "B Cell": ["Ms4a1"],
    "T Cell": ["Cd3d"],
    "NK Cell": ["Klrd1"],
}

markers_list = pd.Series([b for a in celltype_markers for b in celltype_markers[a]])
markers_list[~markers_list.isin(adata_macro.var_names)]

for res in [0.6]:
    res_key = f"{CLUSTER_KEY}_{res:.1f}"
    clear_uns(adata_macro, "colors")
    plot_violinplot(adata_macro, res_key, celltype_markers, bracket_fontsize=8)

In [ ]:
# remove likely doublets
adata_macro = adata_macro[
    ~adata_macro.obs[f"{CLUSTER_KEY}_0.8"].isin([6, 15, 16])
].copy()
print(adata_macro.shape)

In [ ]:
# separate Dendritic Cells
adata_dcs = adata_macro[adata_macro.obs[f"{CLUSTER_KEY}_0.8"].isin([11, 13])].copy()
print(adata_dcs.shape)
clear_adata(adata_dcs, [CELLTYPE_ADDON, "colors", "dendrogram"])
adata_dcs.write_zarr(SUBSETS_DIR / "dendritic")

In [ ]:
adata_macro = adata_macro[~adata_macro.obs[f"{CLUSTER_KEY}_0.8"].isin([11, 13])].copy()

### Clustering

In [ ]:
# cluster
cluster = False
resolutions = np.arange(1, 10) / 10

# cluster
if cluster is True:
    Visualize(adata_macro, CELLTYPE_ADDON, input_key=INT_KEY)
    Cluster(adata_macro, CLUSTER_KEY, resolutions, neighbor_key="neighbors_macro")

# plot clusters
for embedding in [f"UMAP_{CELLTYPE_ADDON}", f"LocalMAP_{CELLTYPE_ADDON}"]:
    r, c = 3, 3
    f, axs = plt.subplots(r, c, figsize=(8 * c, 6 * r), layout="constrained")
    axs = axs.flatten()

    for i, res in enumerate(tqdm(resolutions)):
        res_key = f"{CLUSTER_KEY}_{res}"
        cluster_c = color_gen(adata_macro.obs[res_key].cat.categories)
        adata_macro.obs[res_key] = (
            adata_macro.obs[res_key].astype(int).astype("category")
        )

        sc.pl.embedding(
            adata_macro,
            basis=embedding,
            color=res_key,
            ax=axs[i],
            show=False,
            legend_loc="on data",
            legend_fontoutline=2,
            legend_fontsize=15,
            palette=cluster_c,
        )
        axs[i].annotate(
            f"n = {adata_macro.shape[0]}",
            size=15,
            fontweight="bold",
            xy=(0.98, 0.02),
            xycoords="axes fraction",
            horizontalalignment="right",
            verticalalignment="bottom",
        )

In [ ]:
# metadata checks
embedding = f"LocalMAP_{CELLTYPE_ADDON}"

f = plt.figure(figsize=(9, 12), layout="constrained")
check_integration(
    adata_macro,
    "Dataset",
    f,
    embeddings=[embedding],
    nrow=3,
)

f = plt.figure(figsize=(7, 10), layout="constrained")
check_integration(
    adata_macro,
    "Diet",
    f,
    embeddings=[embedding],
    nrow=2,
)

f = plt.figure(figsize=(9, 15), layout="constrained")
check_integration(
    adata_macro,
    "Age",
    f,
    embeddings=[embedding],
    nrow=4,
)

In [ ]:
plot_cluster_trees(
    adata_macro,
    [f"{CLUSTER_KEY}_{res}" for res in resolutions],
    threshold=0.05,
    node_size_range=(200, 2000),
    x_spacing=2,
    y_spacing=2,
)

In [ ]:
# barplot counts
for col in adata_macro.obs.columns:
    if CLUSTER_KEY in col:
        adata_macro.obs[col] = adata_macro.obs[col].astype(int).astype("category")

r, c = 3, 3
f, axs = plt.subplots(r, c, figsize=(10 * c, 5 * r), layout="constrained")
axs = axs.flatten()
for i, res in enumerate(resolutions):
    res_key = f"{CLUSTER_KEY}_{res}"
    plot_cluster_counts(adata_macro, res_key, ax=axs[i])

# percentage breakdowns
for col in ["Dataset"] + METADATA[:2]:
    f, axs = plt.subplots(r, c, figsize=(10 * c, 6 * r), layout="constrained")
    axs = axs.flatten()
    for i, res in enumerate(resolutions):
        res_key = f"{CLUSTER_KEY}_{res}"
        plot_cluster_stackedbarplot(adata_macro, res_key, col, pct=True, ax=axs[i])
    f.suptitle(col + " Split", size=30)

# Exploration

### Markers

In [ ]:
embedding = f"UMAP_{CELLTYPE_ADDON}"

f = clustree(
    adata_macro,
    [f"{CLUSTER_KEY}_{res}" for res in SELECT_RES],
    edge_weight_threshold=0.01,
    x_spacing=5,
    y_spacing=0.5,
    node_size_range=(300, 2000),
)
f.set_figheight(5)

for res in tqdm(SELECT_RES):
    res_key = f"{CLUSTER_KEY}_{res:.1f}"
    cluster_c = color_gen(adata_macro.obs[res_key].cat.categories)
    adata_macro.obs[res_key] = adata_macro.obs[res_key].astype(int).astype("category")

    f, ax = plt.subplots(1, 1, figsize=(15, 12))
    sc.pl.embedding(
        adata_macro,
        basis=embedding,
        color=res_key,
        ax=ax,
        show=False,
        legend_loc="on data",
        legend_fontoutline=1.5,
        legend_fontsize=20,
        palette=cluster_c,
        size=15,
    )
    ax.annotate(
        f"n = {adata_macro.shape[0]}",
        size=15,
        fontweight="bold",
        xy=(0.98, 0.02),
        xycoords="axes fraction",
        horizontalalignment="right",
        verticalalignment="bottom",
    )


# for combos in list(itertools.combinations(SELECT_RES,2)):
#     f = clustree(
#         adata_macro,
#         [f"{CLUSTER_KEY}_{res}" for res in combos],
#         scatter_reference=embedding,
#         edge_weight_threshold=0.01,
#         node_size_range=(200, 300),
#         edge_width_range=(0.1, 2.0),
#         graph_plot_kwargs={"font_color": "black", "font_size": 10, "alpha": 0.75},
#     )

In [ ]:
# fmt: off
mac_paper_markers = {
    "Arg1 FALC Macs": ["Arg1", "Thbs1", "Lyz1", "Cd5l", "Mertk", "Timd4", "Fn1", "Clec4d", "Marco", "Itpr1", "Slpi", "Alox15", "Cd38"],
    "Lipo VAMs": ["Maf", "Lpl", "Mrc1", "H2-Ab1", "Lipa", "Retnla", "Ly6e"],
    "Septal Macs": ["Mrc1", "Maf", "Lyve1", "Cd209f", "Cd74", "Mgl2"],
    "Classical Mono": ["Ccr2", "Ly6c1", "Ly6c2", "Cd226", "Egfr"],
    "Non-classical Mono": ["Plac8"],
    "Egfr ligands": ["Egf", "Tgfa", "Hbegf", "Areg", "Btc", "Epgn"],
    "OK-LAM": ["Gpnmb", "Mmp12", "Cd36"],
    "Bad-LAM": ["Trem2", "Ctsd", "Ctsl", "Cd9", "Fabp5"],
    "Tissue-resident VAM": ["Lyve1", "Cd163", "Ednrb", "F13a1", "Apoe", "Cd36", "Ccl24", "Folr2"],
    "Dividing GH2025": ["Mki67", "Top2a", "Ube2c"],
    "Neural GH2025": ["Siglec1", "Maoa"],
    "Kohda2025": ["Clec4e"],
    "Aging macs GH2025": ["Cxcl13", "C3", "Cd55", "Ly6e", "Ccl8", "Colq", "Mmp9"],
    "Other GH2025": ["Cd209a", "Itgam", "Itgax", "Zbtb46", "Spp1", "Rsad2"],
    "Lipophagy": ["Map1lc3b", "Sqstm1", "Atg16l1", "Sra1", "Cd36", "Sirt6", "Pnpla2", "Acadm", "B4galnt1", "Hadh", "Inpp5d", "Soat1", "Ip6k1", "Hacl1", "Echs1", "Dgkz", "Hadhb"],
    "Efferocytosis": ["Mertk", "Lrp1", "Havcr1", "Timd4", "Tyro3", "Axl", "Cd36", "Cx3cr1", "Wdfy3", "C1qa", "C1qb", "C1qc", "Nr1h3", "Nr1h2", "Pparg"],
}  # Ereg not present in filtered sample
unique_markers = pd.Series(
    list(set([i for j in mac_paper_markers.values() for i in j]))
)
assert np.all(unique_markers.isin(adata_macro.var_names))


mac_markers = {
    "Ccr genes": ["Ccr1", "Ccr2", "Ccr3", "Ccr4", "Ccr5", "Ccr6", "Ccr7", "Ccr9", "Ccr10"],
    "Common IFN response": ["Irf9", "Irf7", "Ifi35", "Ifnar2", "Isg20", "Isg15", "Ifit1", "Ifit2", "Ifih1", "Ifnar1", "Ifngr2", "Cxcl9", "Oas1a", "Mx1"],
    "Type I IFN": ["Cxcl10", "Isg15", "Mx1", "Irf3", "Irf7", "Il6", "Setdb2"],
    "Type II IFN": ["Irf1", "Stat1", "Mx1", "Bst2", "Mafb"],
    "TGFb response": ["Pfkl", "Arg1", "Cebpa", "Id3", "Retnla", "F13a1", "Tgfbr1", "Tgfbr2", "Pdcd1", "Smad3", "Smad7", "Cx3cr1", "Gcnt2", "Pmepa1", "Runx3", "Axl", "F11r", "Apoe", "Mmp14"],
    "TNF response": ["Nfkb1", "Nfkbia", "Jun", "Fos", "Mapk1", "Mapk3", "Mapk8", "Mapk10", "Srebf2", "Il1a", "Il1b", "Il18", "Ccl5"],
    "Type 1 activation": ["Cd86", "Cd40", "Cxcl16", "Cxcl9", "Il15ra", "Il17ra", "Icam1", "Vcam1", "Nfkb1", "Rela", "Rps6ka2", "Tank", "Ripk2", "Il15", "Il23a", "Irf1", "Slc7a2", "Slc12a4", "Slc1a4", "Slc39a14", "Slc3a2", "Slc4a7", "Csf2ra", "Csf2rb", "Csf2rb2"],
    "Type 2 activation": ["Folr2", "Mertk", "Arg1", "Chil3", "Retnla", "Aldh1a2", "Ucp1", "Lpxn", "Dhrs3", "Mical1", "Dnmt3a", "Jun", "Gab1", "P2ry1", "Gab2", "Glul", "Cpt1a", "Acadm", "B4galnt1", "Hadh", "Inpp5d", "Soat1", "Ip6k1", "Hacl1", "Echs1", "Dgkz", "Hadhb", "Parp1", "Ptpn22", "Cd300a", "Cd84", "Csf1", "Csf1r"],
    "Non-specific activation": ["Ccl7", "Ccl17", "Ccl22", "Ccl24", "Cd38", "Cd44", "Cd83", "Csf3r", "Alcam"],
    "Selected markers": ["Gas6", "Il1rn", "S100a4", "Maf1", "Col1a2", "Col5a2"],
    "General IFN/TGFb": ["Ifngr1", "Ifnar1", "Tgfbr2", "Il33", "Il31ra", "Il17ra", "Il17rb"],
}
unique_markers = pd.Series(list(set([i for j in mac_markers.values() for i in j])))
# print(unique_markers[~unique_markers.isin(adata_macro.var_names)].tolist())
# print((~unique_markers.isin(adata_macro.var_names)).sum())
assert np.all(unique_markers.isin(adata_macro.var_names))
# fmt: on

In [ ]:
for col in adata_macro.obs.columns:
    if CLUSTER_KEY in col:
        adata_macro.obs[col] = adata_macro.obs[col].astype(str).astype("category")
        # adata_macro.obs[col] = adata_macro.obs[col].astype(int).astype("category")

for res in SELECT_RES:
    res_key = f"{CLUSTER_KEY}_{res:.1f}"
    de_key = f"{DE_KEY}_{res:.1f}"
    sc.pl.rank_genes_groups_stacked_violin(
        adata_macro,
        groupby=res_key,
        key=de_key,
        var_names=mac_markers | mac_paper_markers,
        # values_to_plot="logfoldchanges",
        # cmap="bwr",
        # colorbar_title="log fold change",
        dendrogram=False,
        title=f"{res_key}_DEGs",
        vmin=-3,
        vmax=3,
    )

    # plot_violinplot(adata_macro, res_key, mac_markers | mac_paper_markers)

In [ ]:
# counts/breakdowns
for col in adata_macro.obs.columns:
    if CLUSTER_KEY in col:
        adata_macro.obs[col] = adata_macro.obs[col].astype(int).astype("category")

r, c = 1, 3

# count barplots
f, axs = plt.subplots(r, c, figsize=(10 * c, 5 * r), layout="constrained")
axs = axs.flatten()
for i, res in enumerate(SELECT_RES):
    res_key = f"{CLUSTER_KEY}_{res}"
    plot_cluster_counts(adata_macro, res_key, ax=axs[i])

# percentage breakdowns
for col in ["Dataset", "Diet"]:
    f, axs = plt.subplots(r, c, figsize=(10 * c, 6 * r), layout="constrained")
    axs = axs.flatten()
    for i, res in enumerate(SELECT_RES):
        res_key = f"{CLUSTER_KEY}_{res}"
        plot_cluster_stackedbarplot(adata_macro, res_key, col, pct=True, ax=axs[i])
    f.suptitle(col + " Split", size=30)

### DEGs

In [ ]:
# original subclusters
RUN_DEG = False

# change to string for DEGs
for col in adata_macro.obs.columns:
    if CLUSTER_KEY in col:
        adata_macro.obs[col] = adata_macro.obs[col].astype(str).astype("category")

for res in SELECT_RES:
    res_key = f"{CLUSTER_KEY}_{res:.1f}"
    de_key = f"{DE_KEY}_{res:.1f}"
    n_clust = len(adata_macro.obs[res_key].unique())

    # calculate DEGs
    if RUN_DEG is True:
        clear_uns(adata_macro, res_key)
        sc.tl.rank_genes_groups(
            adata_macro,
            groupby=res_key,
            key_added=de_key,
            use_raw=False,
            layer="normalized",
            method="wilcoxon",
        )

    # plot DEGs
    N_genes = 20
    top_genes = {}
    for group, df in sc.get.rank_genes_groups_df(
        adata_macro, group=None, key=de_key
    ).groupby("group"):
        top_genes[group] = df["names"][:N_genes].tolist()

    f, ax = plt.subplots(1, 1, figsize=(n_clust * 5, 5), layout="constrained")
    sc.pl.rank_genes_groups_dotplot(
        adata_macro,
        groupby=res_key,
        key=de_key,
        # n_genes=20,
        var_names=top_genes,
        # standard_scale="var",
        values_to_plot="logfoldchanges",
        cmap="bwr",
        colorbar_title="log fold change",
        ax=ax,
        title=f"{res_key}_DEGs",
        vmin=-4,
        vmax=4,
    )

    sc.pl.rank_genes_groups_heatmap(
        adata_macro,
        key=de_key,
        groupby=res_key,
        layer="normalized",
        n_genes=50,
    )

### Specific comparisons

In [ ]:
temp_adata = dict([
    (cond, adata_macro[(adata_macro.obs["Diet"] == cond) & adata_macro.obs["leiden_macro_0.6"].isin(['1', '5'])])
    for cond in adata_macro.obs["Diet"].unique()
])
print(temp_adata)
pd.crosstab(adata_macro.obs["Diet"], adata_macro.obs["leiden_macro_0.6"])[["1", "5"]]

In [ ]:
res_key = "leiden_macro_0.6"
for cond in temp_adata:
    sc.tl.rank_genes_groups(temp_adata[cond], "leiden_macro_0.6")

    # plot DEGs
    N_genes = 20
    top_genes = {}
    for group, df in sc.get.rank_genes_groups_df(
        temp_adata[cond], group=None
    ).groupby("group"):
        top_genes[group] = df["names"][:N_genes].tolist()

    f, ax = plt.subplots(1, 1, figsize=(10, 5), layout="constrained")
    sc.pl.rank_genes_groups_dotplot(
        temp_adata[cond],
        groupby=res_key,
        # n_genes=20,
        var_names=top_genes,
        # standard_scale="var",
        values_to_plot="logfoldchanges",
        cmap="bwr",
        colorbar_title="log fold change",
        ax=ax,
        title=f"{res_key} DEGs {cond} (1 & 5 only)",
        vmin=-4,
        vmax=4,
    )

# Save/Load

In [37]:
# save
clear_adata(adata_macro, ["dendrogram", "colors"])
adata_macro.write_zarr(SUBSETS_DIR / (CELLTYPE_ADDON + '_reintegrated.zarr'))

In [ ]:
# load
adata_macro = ad.read_zarr(SUBSETS_DIR / (CELLTYPE_ADDON + '.zarr'))
adata_macro

In [ ]:
# adata_macro.obsm[f'LocalMAP_{CELLTYPE_ADDON}_global'] = adata_macro.obsm[f'LocalMAP_{CELLTYPE_ADDON}'].copy()
# adata_macro.obsm[f'UMAP_{CELLTYPE_ADDON}_global'] = adata_macro.obsm[f'UMAP_{CELLTYPE_ADDON}'].copy()

# adata_macro.obsp[f'neighbors_{CELLTYPE_ADDON}_global_connectivities'] = adata_macro.obsp[f'neighbors_{CELLTYPE_ADDON}_connectivities'].copy()
# adata_macro.obsp[f'neighbors_{CELLTYPE_ADDON}_global_distances'] = adata_macro.obsp[f'neighbors_{CELLTYPE_ADDON}_distances'].copy()

# adata_macro.uns[f'neighbors_{CELLTYPE_ADDON}'][f'connectivities_key'] = f'neighbors_{CELLTYPE_ADDON}_global_connectivities'
# adata_macro.uns[f'neighbors_{CELLTYPE_ADDON}'][f'distances_key'] = f'neighbors_{CELLTYPE_ADDON}_global_distances'
# adata_macro.uns[f'neighbors_{CELLTYPE_ADDON}_global'] = adata_macro.uns[f'neighbors_{CELLTYPE_ADDON}'].copy()

# del adata_macro.obsm[f'UMAP_{CELLTYPE_ADDON}'], adata_macro.obsm[f'LocalMAP_{CELLTYPE_ADDON}'], adata_macro.obsp[f'neighbors_{CELLTYPE_ADDON}_connectivities'], adata_macro.obsp[f'neighbors_{CELLTYPE_ADDON}_distances'], adata_macro.uns[f'neighbors_{CELLTYPE_ADDON}']

In [ ]:
# del adata_macro.obsm["UMAP"], adata_macro.obsm["X_pca"]
# rename_obsm(adata_macro, "INT", prefix="global_")
# rename_obsm(adata_macro, "PCA", prefix="global_")
# rename_obsm(adata_macro, "Dataset", replace="all-Dataset", filter="hvg")
# rename_obsm(adata_macro, "Identifier", replace="all-Identifier", filter="hvg")
# rename_obsm(adata_macro, "_hvg", replace="-hvg")
# rename_obsm(adata_macro, "PCA", replace="PCA-all", filter="hvg")
# rename_obsm(adata_macro, "none", replace="none-all", filter="hvg")

# # list(adata_macro.obsm)